# Lab 3 — Inspect & Talk to Models

**Day 1 Morning | ~60 minutes | CPU only, no API key**

---

In Lab 2 you sent prompts to a model you could not see. It lived on someone else's servers, you handed it a list of messages, and text came back. That works fine until the day it doesn't, and then you are debugging a black box.

Today we open one up. We will load a small model onto this machine, read its configuration, look at its actual weight tensors, watch the memory it allocates while it generates, and then build a chat loop out of the pieces. Nothing here needs a GPU or an API key.

The model is `Qwen2.5-0.5B-Instruct`. It is small enough to run on a CPU in a classroom and modern enough to use the same architecture tricks as the models you will actually deploy.

One warning before you start: **do not trust a word this model says.** At 0.5B parameters it will state things that are wrong with total confidence, and it does so several times in this notebook. We are here to study the machinery — the shapes, the memory, the prompt format — not to learn about quantization from it. When an answer looks wrong, it probably is, and that is not a bug in your setup.

## What you will walk out with

1. Attention is three vectors per token, and once you know what they are, half the config file explains itself.
2. Grouped Query Attention is the single most important number for serving capacity, and you can see it in the weight shapes.
3. The KV cache is not an optimization. Without it, generation is unusably slow, and you can measure exactly how slow.
4. Temperature and top-p reshape a probability distribution. You will look at the distribution itself, not just the text.
5. A chatbot has no memory. You build the illusion by hand, and it costs you tokens on every turn.

## The route

```
Part A                Part B              Part C                Part D
────────────────      ──────────────      ────────────────      ──────────────────
What is inside   →    The KV cache   →    Generation      →     Multi-turn chat
the model             (and why it          controls              (history by hand,
(Q, K, V, heads,       makes or broke      (probabilities,        then as a class)
 GQA, the config)      your latency)        temperature, top-p)

~20 min               ~12 min             ~15 min               ~13 min
```

Each part ends with a **Checkpoint**. If you cannot answer it, scroll back before moving on. The parts build on each other.

**Coming from Lab 2:** that was a hosted model behind an API. This is a local one you can take apart. The history list you managed by hand in Lab 2 shows up again in Part D, and this time it becomes a class.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} "transformers>=5" torch accelerate

In [ ]:
# Small, modern, instruction-tuned. About 1 GB on first download.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Model for this lab: {MODEL_ID}")
print("CPU is fine. You do not need a GPU until Lab 4.")

---

## What is a model, physically?

When you call `from_pretrained(...)`, three things happen:

1. A **config file** downloads. It is JSON, a few dozen lines, and it describes the shape of the network: how many layers, how wide, how many attention heads.
2. The **weights** download. These are the numbers learned during training, roughly a billion bytes for this model.
3. Both get **loaded into RAM** so Python can do arithmetic with them.

That is the whole thing. There is no process running somewhere, no state, no memory of you. Every call starts from nothing. Whatever the model appears to remember is something you put in the prompt yourself, and Part D is where that becomes uncomfortably concrete.

## Why read the config at all?

Because these numbers decide things you will otherwise find out the expensive way.

| You need to decide | The number that decides it |
|---|---|
| Which GPU to rent | hidden size, layers, dtype → weight memory |
| How many users one GPU serves | key/value heads → KV cache per user |
| How much room RAG context gets | context window minus your system prompt |
| Whether to quantize at all | BF16 footprint vs your available VRAM |
| Which of two models to pick | all of the above, side by side |

### One caveat: you cannot always look

| What you want | Open weights (Qwen, Llama, Gemma, Mistral) | API only (GPT-5, Claude, Gemini) |
|---|---|---|
| Layers, hidden size, GQA ratio | `model.config` | not published |
| Weights, for quantizing or tuning | you have the files | the provider holds them |
| Context window | `config.max_position_embeddings` | documented |
| KV cache behaviour | inspectable | not published |
| Cost per call | your own compute bill | per-token pricing |

If you are self-hosting, you need to be able to inspect. If you are calling an API, you design around what the provider documents and let them worry about the hardware.

---

# Part A — What is inside the model

**~20 minutes**

We are going to read a config file in a minute. But `num_key_value_heads` means nothing until you know what a key and a value are, so let's start there.

## Attention, in three vectors

Every token in your prompt produces three vectors. Not one. Three, and they have different jobs.

- The **query** is what this token is looking for.
- The **key** is what this token advertises about itself.
- The **value** is what this token actually contributes to whoever looks at it.

To work out how much token 5 should care about token 2, the model takes token 5's *query* and token 2's *key* and multiplies them together. A big number means a strong match. Do that against every earlier token, squash the scores through a softmax so they add up to 1, and use them as mixing weights over all the *values*. The blend that comes out is what attention produced for token 5.

If it helps: you are searching a library. Your **query** is what you want. Each book's **key** is the label on its spine. The **value** is what is printed inside. You match your query against the spines, then read the contents of the books that matched, weighted by how well they matched.

## Heads

One set of queries, keys and values gives the model exactly one way to relate tokens to each other. That is limiting, so it runs several in parallel, each with its own projection. One head might end up tracking which noun a pronoun refers to, another might follow the structure of a list. Nobody assigns them these jobs; they fall out of training.

These are **attention heads**, and every layer has its own full set of them.

> ### 🖥️ On screen right now
>
> The [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) is showing GPT-2 doing exactly this. Type a sentence, then click into an attention head:
>
> - The three stacked columns per token are **Query, Key, Value**.
> - The lines between tokens are attention scores: query · key, after the softmax.
> - Switch heads and watch the lines move. Different head, different relationships.
>
> GPT-2 is a 2019 model. It has 12 heads, and 12 matching sets of keys and values. The model you are about to load handles this differently, and that difference is the most important number in this lab.

#### ✏️ Predict before you run

Three guesses. Write them down, they take ten seconds, and being wrong is the point.

1. A "0.5B" model has about 500 million parameters. More or fewer than 10 transformer layers?
2. How many tokens fit in its context window?
3. The query projection and the key projection — same size, or different?

In [ ]:
# Cell A1 — Load the tokenizer and the model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
config = model.config

print("Loaded. Weights are in RAM.")

First run downloads about 1 GB and takes a couple of minutes. After that it is cached and loading is nearly instant.

Now the config. Six numbers.

In [ ]:
# Cell A2 — The six numbers that describe this model's shape
print(f"Hidden size     : {config.hidden_size}")
print(f"Layers          : {config.num_hidden_layers}")
print(f"Attention heads : {config.num_attention_heads}")
print(f"Key/value heads : {config.num_key_value_heads}")
print(f"Vocab size      : {config.vocab_size:,}")
print(f"Context window  : {config.max_position_embeddings:,} tokens")

**Check your predictions.**

- **24 layers.** Most people guess around 12 for a model this size. This one is deep and narrow: more layers than you would expect, each one thinner than you would expect. That is a deliberate trade. Depth gives the model more sequential steps to work with; narrowness keeps the compute per step down. It also means latency scales with those 24 sequential layers, and you cannot parallelize your way out of that.
- **32,768 tokens** of context. Somewhere around 25,000 words. Generous for a 0.5B model.
- **Hidden size 896, 14 attention heads.** 896 ÷ 14 = 64, which is the size of each head. That number shows up again shortly.

And then the odd one: **14 attention heads, but only 2 key/value heads.** If every head needs a query, a key and a value, how do 14 heads share 2 sets of keys and values?

Hold that thought. First, some arithmetic you will care about in Lab 4.

In [ ]:
# Cell A3 — Parameters and what they cost in memory
params = sum(p.numel() for p in model.parameters())

print(f"Parameters   : {params:,}  ({params/1e9:.2f} B)")
print(f"BF16 weights : ~{params * 2 / 1e9:.2f} GB   (2 bytes per parameter)")
print(f"INT4 weights : ~{params * 0.5 / 1e9:.2f} GB   (0.5 bytes -- this is Lab 4)")

Weights are only part of the bill. Actual RAM while serving is weights **plus** activations **plus** the KV cache, and that last one is the one that surprises people. Part B measures it.

Next, the structure itself. One decoder layer, printed.

In [ ]:
# Cell A4 — What one transformer layer actually contains
print(f"The stack is {len(model.model.layers)} of these:\n")
print(model.model.layers[0])

Attention block, MLP block, two layer norms. That is it, and layers 1 through 23 are identical in structure. There is no special "early" layer or "reasoning" layer. One design, stacked 24 times, each copy with its own learned weights.

Inside `self_attn` are the four projections that produce the vectors we talked about: `q_proj` makes queries, `k_proj` makes keys, `v_proj` makes values, and `o_proj` mixes the heads back together afterwards.

**Before you run the next cell:** all four take the same 896-dimensional input. Do you expect all four to produce 896 numbers back?

In [ ]:
# Cell A5 — The shapes of the four attention projections
attn = model.model.layers[0].self_attn

for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    out_features, in_features = getattr(attn, name).weight.shape
    print(f"{name:8s}  {in_features} -> {out_features}")

There it is. `q_proj` produces 896 numbers. `k_proj` and `v_proj` produce **128**.

Do the arithmetic:

- 14 query heads × 64 per head = **896** ✅
- 2 key/value heads × 64 per head = **128** ✅

The model really does build 14 queries and only 2 keys and values per token. This is not a bug or a compression applied afterwards — the network was trained this way, and the smaller tensors are baked into the architecture.

In [ ]:
# Cell A6 — Confirm the arithmetic
head_dim = config.hidden_size // config.num_attention_heads

print(f"head_dim = {config.hidden_size} / {config.num_attention_heads} = {head_dim}")
print(f"query  side: {config.num_attention_heads} heads x {head_dim} = {config.num_attention_heads * head_dim}")
print(f"key/val side: {config.num_key_value_heads} heads x {head_dim} = {config.num_key_value_heads * head_dim}")

## Why would anyone build it that way?

There is a short history here, and it is worth knowing because it explains most modern model configs.

**Multi-Head Attention (MHA)** is the original, from the 2017 *Attention Is All You Need* paper. Every attention head gets its own query, its own key, its own value. GPT-2 does this: 12 heads, 12 sets of keys and values. It works well and nobody complained for a while.

The complaint arrived when people started *serving* these models. During generation you have to keep the keys and values for every token you have seen so far (Part B explains why). With MHA that stored tensor is proportional to the number of heads, and it turns out to be the thing that fills your GPU, not the weights.

**Multi-Query Attention (MQA)** was Noam Shazeer's 2019 answer: keep all the query heads, but let them share a *single* key/value head. The memory problem goes away. Unfortunately so does some of the quality, and training could get unstable.

**Grouped Query Attention (GQA)** is the compromise, from Ainslie et al. at Google in 2023 (*GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*, EMNLP 2023). Instead of one shared key/value head or one per query head, you make **groups**. Several query heads share one key/value head, and you pick how many.

```
MHA   14 query heads, 14 key/value heads     full memory cost, best quality
MQA   14 query heads,  1 key/value head      cheapest, some quality loss
GQA   14 query heads,  2 key/value heads     this model. 7 queries per group.
```

Qwen2.5-0.5B puts its 14 query heads into 2 groups of 7. Heads 0–6 all read the same keys and values; heads 7–13 read the other set. They still have their own distinct queries, so they still attend differently — they just consult a shared reference.

The paper's finding was that quality stays close to MHA while memory drops toward MQA. That was a good enough deal that essentially every production model adopted it: Llama 3, Mistral, Qwen 2.5, Gemma 2. When you look at a new model's config and see key/value heads below attention heads, this is what you are seeing.

Here is what that decision is worth in bytes.

In [ ]:
# Cell A7 — What GQA saves, per token of context
bytes_per_number = 2        # bfloat16
k_and_v = 2                 # we store a key AND a value

gqa_bytes = k_and_v * config.num_hidden_layers * config.num_key_value_heads * head_dim * bytes_per_number
mha_bytes = k_and_v * config.num_hidden_layers * config.num_attention_heads * head_dim * bytes_per_number

print(f"This model (GQA, {config.num_key_value_heads} kv heads) : {gqa_bytes:,} bytes per token")
print(f"If it used MHA ({config.num_attention_heads} kv heads)   : {mha_bytes:,} bytes per token")
print(f"Saving: {mha_bytes / gqa_bytes:.0f}x")

Seven times less memory per token of conversation, for the same model quality. Cell B4 turns that multiplier into an actual user count.

We will spend that number properly in Part B. First, a sanity check against the model you just saw in the browser.

In [ ]:
# Cell A8 — GPT-2 (2019) next to Qwen2.5 (2024). Config only, no weights downloaded.
from transformers import AutoConfig

gpt2 = AutoConfig.from_pretrained("gpt2")

rows = [
    ("hidden size",      gpt2.hidden_size,             config.hidden_size),
    ("layers",           gpt2.num_hidden_layers,       config.num_hidden_layers),
    ("attention heads",  gpt2.num_attention_heads,     config.num_attention_heads),
    ("key/value heads",  getattr(gpt2, "num_key_value_heads", "not defined"),
                         config.num_key_value_heads),
    ("context window",   gpt2.max_position_embeddings, config.max_position_embeddings),
    ("vocab size",       gpt2.vocab_size,              config.vocab_size),
]

print(f"{'':18s}{'GPT-2':>16s}{'Qwen2.5-0.5B':>16s}")
print("-" * 50)
for label, a, b in rows:
    print(f"{label:18s}{str(a):>16s}{str(b):>16s}")

Read the third row from the bottom of that output: GPT-2 has no `num_key_value_heads` **at all**. The field does not exist in its config, because in 2019 there was nothing to configure. Every head had its own keys and values and that was the only option on the menu.

The other gap worth noticing is context: **1,024 tokens versus 32,768**. GPT-2 could hold about 750 words. That is not a long conversation, and it is a large part of why the 2019 models felt like toys.

Two models, five years apart, same basic architecture. The differences are almost entirely about making the thing servable.

#### ✅ Checkpoint A

Answer these out loud before continuing:

- [ ] What are the three vectors every token produces, and what does each one do?
- [ ] Why is `k_proj` smaller than `q_proj` in this model?
- [ ] How many query heads share one key/value head here?
- [ ] What problem was GQA invented to solve? (Hint: it was not accuracy.)

<details>
<summary>Answers</summary>

**Query, key, value.** Query is what a token is looking for, key is what it advertises, value is what it contributes when attended to. Attention scores come from queries against keys; the output is a weighted mix of values.

**`k_proj` is smaller** because there are only 2 key/value heads versus 14 query heads. 2 × 64 = 128 outputs instead of 14 × 64 = 896.

**Seven.** 14 query heads in 2 groups.

**Memory at serving time.** Specifically the KV cache, which grows with every token of context and every concurrent user. GQA made long-context serving affordable. Quality was the thing GQA was trying not to lose, not the thing it was trying to improve.

</details>

---

# Part B — The KV cache

**~12 minutes**

This part exists because "KV cache" gets thrown around constantly in deployment conversations and rarely gets explained. It is also the reason the previous section mattered.

## The problem

Generation is a loop. The model reads your prompt and produces one token. You append that token and run the model again. It produces another. Repeat until it emits a stop token.

Run that naively and the loop is brutal. To produce token 300, you re-run the entire network over all 299 previous tokens. Then for token 301 you re-run it over 300. The total work grows with the square of the sequence length, and almost all of it is work you already did.

## The fix

Look back at what attention actually needs. For a new token, you need:

- that token's **query** (new, must be computed)
- the **keys** and **values** of every previous token

And here is the thing: the keys and values of token 2 were computed from token 2. Token 2 has not changed. It is still sitting there being the same token it was 300 steps ago, producing the same key and the same value.

So you keep them. Compute each token's key and value once, store them, and reuse them on every subsequent step. That store is the **KV cache**, and it turns each generation step into work proportional to the sequence length rather than the square of it.

The cost is memory, and the amount of memory is `num_key_value_heads`. Which is why we spent Part A on GQA.

Let's look at the real thing.

In [ ]:
# Cell B1 — Run one forward pass and inspect the cache it produced
inputs = tokenizer("The capital of France is", return_tensors="pt")
outputs = model(**inputs, use_cache=True)
cache = outputs.past_key_values

print(f"Cache object   : {type(cache).__name__}")
print(f"Layers cached  : {len(cache.layers)}")
print(f"Tokens in input: {inputs['input_ids'].shape[1]}")
print()
print(f"Layer 0 keys   : {tuple(cache.layers[0].keys.shape)}")
print(f"Layer 0 values : {tuple(cache.layers[0].values.shape)}")

## Read that shape one dimension at a time

You should see `(1, 2, 5, 64)`.

| Position | Value | What it is |
|---|---|---|
| batch | **1** | one sequence at a time. Serve 8 users at once and this becomes 8. |
| heads | **2** | 🎯 **the key/value heads.** This is GQA, in memory, right now. |
| tokens | **5** | one entry per token processed so far. This is the axis that grows. |
| head_dim | **64** | 896 ÷ 14, the width of a single head. |

That **2** is the payoff from Part A. If this model used MHA it would be a 14 there, and everything downstream — memory per user, users per GPU, your monthly bill — would be seven times worse.

And there are 24 of these, one per layer, keys and values both.

The third number is the one that moves. Let's prove it.

In [ ]:
# Cell B2 — More tokens in, bigger cache
for text in ["The capital of France is",
             "The capital of France is Paris, and the capital of Japan is",
             "The capital of France is Paris, and the capital of Japan is Tokyo, and the capital of Kenya is"]:
    ids = tokenizer(text, return_tensors="pt")
    c = model(**ids, use_cache=True).past_key_values
    n_tokens = ids["input_ids"].shape[1]
    total_bytes = gqa_bytes * n_tokens
    print(f"{n_tokens:3d} tokens -> layer 0 keys {tuple(c.layers[0].keys.shape)} -> {total_bytes/1024:7.1f} KB across all layers")

Linear, exactly as advertised. Every token you add costs the same fixed number of bytes, forever, for as long as that conversation stays open.

Now the part that makes it worth the memory.

### Measuring what the cache is worth

The next cell generates the same 30 tokens twice: once with the cache on, once with it off. Same model, same prompt, same output.

⚠️ **The second run is deliberately slow.** Expect somewhere between 30 and 90 seconds depending on your machine. That wait *is* the lesson, so let it finish.

In [ ]:
# Cell B3 — Generation with and without the cache
import time

notes = "Deployment notes: " + ("Latency, throughput, cost, and memory all matter when serving models. " * 15) + "\nSummarize the concerns in one sentence."
formatted = tokenizer.apply_chat_template(
    [{"role": "user", "content": notes}], tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(formatted, return_tensors="pt")
print(f"Prompt length: {inputs['input_ids'].shape[1]} tokens. Generating 30 new tokens each way.\n")

timings = {}
for use_cache in [True, False]:
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=30, do_sample=False,
                       use_cache=use_cache, pad_token_id=tokenizer.eos_token_id)
    timings[use_cache] = time.time() - start
    print(f"use_cache={str(use_cache):5s} : {timings[use_cache]:6.1f} s")

print(f"\nThe cache made it {timings[False] / timings[True]:.0f}x faster.")

That is the whole argument. The KV cache is not a tuning flag you turn on when you have time to optimize; without it, generation is not a viable product. Every serving stack you will meet — vLLM in Lab 5, TGI, llama.cpp, the API you called in Lab 2 — is built around managing this cache well.

Which brings us to the number you will actually be asked about in a capacity meeting.

In [ ]:
# Cell B4 — How many users fit on one GPU?
gpu_gb = 16          # a typical mid-range inference GPU
weights_gb = params * 2 / 1e9
available_gb = gpu_gb - weights_gb

print(f"GPU              : {gpu_gb} GB")
print(f"Model weights    : {weights_gb:.1f} GB")
print(f"Left for cache   : {available_gb:.1f} GB\n")

for ctx in [1_000, 8_000, 32_768]:
    gqa_user_gb = gqa_bytes * ctx / 1e9
    mha_user_gb = mha_bytes * ctx / 1e9
    print(f"{ctx:>6,} tokens of context per user:")
    print(f"    GQA : {gqa_user_gb:5.2f} GB/user -> {available_gb / gqa_user_gb:4.0f} concurrent users")
    print(f"    MHA : {mha_user_gb:5.2f} GB/user -> {available_gb / mha_user_gb:4.0f} concurrent users")

Read the bottom block. At full 32K context, GQA versus MHA is the difference between a few dozen concurrent users and a handful, on identical hardware, running an identically capable model.

This is why `num_key_value_heads` is the first thing an experienced person checks when comparing two open models for self-hosting. Benchmark scores tell you whether the model is good. This tells you whether you can afford to run it.

> A caveat so you are not surprised later: these are clean-room numbers. Real serving stacks add overhead for activations and fragmentation, and then claw memory back with tricks like paged attention, which is vLLM's whole reason for existing. Treat this as the right order of magnitude and the right *shape* of the argument, not a procurement quote.

#### ✅ Checkpoint B

- [ ] Why can the model reuse a token's key and value instead of recomputing them?
- [ ] Which dimension of the cache tensor grows during a conversation?
- [ ] Your cache tensor showed a 2 in the heads position. Where did that 2 come from?
- [ ] Name the two resources the KV cache trades against each other.

<details>
<summary>Answers</summary>

**Because the token did not change.** Keys and values are computed from a token and its position, both of which are fixed once it is in the sequence. Only the newest token needs new keys and values.

**The token axis** (third position). Batch and head count are fixed; head_dim is fixed. Length grows by one per generated token.

**From `num_key_value_heads = 2`,** which is GQA. 14 query heads sharing 2 key/value heads.

**Memory for compute.** You spend RAM to avoid redoing work. Cell B3 measured the compute you save; cell B4 measured the RAM it costs.

</details>

---

# Part C — Generation controls

**~15 minutes**

Three questions this part answers:

- Why does the same prompt sometimes give different answers?
- What does `temperature` actually change?
- Why can you not just send a plain string to an instruct model?

## Generation is not writing, it is picking

The model never composes a sentence. At every step it produces a score for **every single token in its vocabulary** — all 151,936 of them — turns those scores into probabilities, and picks one. Then it does it again.

Two ways to pick:

- **Greedy:** always take the highest-probability token. Same input, same output, every time.
- **Sampling:** draw randomly according to the probabilities. Same input, different output.

Most of what people call "prompt engineering mysteries" is really just this. Let's look at an actual distribution instead of talking about it.

### First, tokens

The model does not see characters or words. It sees integers. The tokenizer is the translator in both directions.

In [ ]:
# Cell C1 — Text goes in, integers come out
sample = tokenizer("Deploying LLMs is mostly plumbing.")

print(sample["input_ids"])
print()
for token_id in sample["input_ids"]:
    print(f"{token_id:>7}  {tokenizer.decode([token_id])!r}")

Notice a few things. The leading space is part of the token, not a separator. Common words are single tokens; less common ones get chopped up. This is why token counts never match word counts, and why your API bill is not proportional to how much you typed.

Now the distribution itself.

In [ ]:
# Cell C2 — What does the model actually believe the next token is?
messages = [{"role": "user", "content": "What is the capital of France? Answer in one word."}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt")

logits = model(**inputs).logits[0, -1]          # scores for the next token only
probs = torch.softmax(logits.float(), dim=-1)   # turn scores into probabilities
top = torch.topk(probs, 5)

print("Top 5 candidates for the next token:\n")
for probability, token_id in zip(top.values, top.indices):
    print(f"  {probability:6.3f}   {tokenizer.decode(token_id)!r}")

About 98% on `'Paris'`. The model is not hedging here, and greedy versus sampling would barely matter on this token.

Two details worth pausing on:

- `' Paris'` with a leading space is a **different token** from `'Paris'`. Both are in the top 5. To the model these are unrelated integers that happen to decode to similar text.
- `'巴黎'` is in there too. That is "Paris" in Chinese, and it is a single token. This is what a 151,936-token multilingual vocabulary buys you, and it is why Qwen's vocab is three times GPT-2's.

Now the interesting bit. Temperature does not change what the model believes. It changes the *shape* of that belief before anyone samples from it.

In [ ]:
# Cell C3 — One set of scores, three temperatures
creative = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Write one word that describes a sunset:"}],
    tokenize=False, add_generation_prompt=True,
)
logits = model(**tokenizer(creative, return_tensors="pt")).logits[0, -1].float()

for temperature in [0.1, 1.0, 1.5]:
    probs = torch.softmax(logits / temperature, dim=-1)   # <- the entire mechanism
    top = torch.topk(probs, 5)
    candidates = "   ".join(f"{tokenizer.decode(i)!r}: {p:.3f}" for p, i in zip(top.values, top.indices))
    print(f"T={temperature}:  {candidates}")

Same logits every time. The only thing that changed is `logits / temperature` before the softmax.

- At **T=0.1** the top couple of tokens hold nearly all the probability. Everything else is rounding error. Sampling from this is almost greedy.
- At **T=1.0** you get the model's honest distribution.
- At **T=1.5** the top token might only hold 2 or 3 percent. The tail — thousands of tokens the model considers unlikely — now collectively owns most of the mass, and the sampler will sometimes reach into it.

That is the whole mechanism. Dividing by a small number exaggerates the gaps between scores; dividing by a large number flattens them. Temperature is a confidence dial on a distribution the model already committed to.

> **Why high temperature produces nonsense:** it is not that the model gets creative. It is that you gave the sampler permission to pick tokens the model thought were bad. One unlucky pick early in a sentence, and every token after it is conditioned on a mistake.

Time to see it in text. First, one small helper.

In [ ]:
# Cell C4 — A helper so the next few cells stay short
def generate(prompt_text, max_new_tokens=60, **kwargs):
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt")
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id, **kwargs)

    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

print("generate() defined. It formats, tokenizes, generates, and returns only the new tokens.")

The `prompt_len` slice matters: `model.generate` returns the prompt *and* the continuation. Without that slice you would print your own question back at yourself.

### Greedy is deterministic. Prove it.

In [ ]:
# Cell C5 — Same prompt, greedy, twice
question = "Name three challenges in deploying an LLM to production."

run1 = generate(question, do_sample=False)
run2 = generate(question, do_sample=False)

print("RUN 1:", run1)
print()
print("RUN 2:", run2)
print()
print("Identical?", run1 == run2)

In [ ]:
# Cell C6 — Same prompt, sampling, twice
run3 = generate(question, do_sample=True, temperature=1.0)
run4 = generate(question, do_sample=True, temperature=1.0)

print("RUN 3:", run3)
print()
print("RUN 4:", run4)
print()
print("Identical?", run3 == run4)

`True` and then `False`. That is the difference, stated as plainly as it can be stated.

Do not skim past why this matters in production:

- **Greedy output is testable.** You can write an assertion against it. You can cache it. You can reproduce a bug report.
- **Sampled output is not.** A customer sends you a screenshot of the model saying something wrong, you run the same prompt, and you get something else. Now you are debugging a distribution instead of a value.

For anything structured — JSON, tool calls, classifications, SQL — start at `do_sample=False` and only add randomness if you can defend it.

### Temperature, in text this time

In [ ]:
# Cell C7 — Low temperature versus high, full output
task = "Describe what a database index does, in one sentence."

print("temperature 0.2")
print(" ", generate(task, max_new_tokens=50, do_sample=True, temperature=0.2))
print()
print("temperature 1.5")
print(" ", generate(task, max_new_tokens=50, do_sample=True, temperature=1.5))

The 0.2 answer should read like a textbook definition. The 1.5 answer is a lottery ticket. It might be *better* — more specific, better phrased — because the sampler reached for a word the safe path would never have picked. It might also wander off, over-qualify everything, or quietly stop making sense.

**Run this cell four or five times.** That is the actual lesson, and one run will not show it to you. At 0.2 the answers will be nearly interchangeable. At 1.5 you get a different answer every time, and the spread in quality between the best and worst is the risk you are accepting when you turn the dial up.

Unreliability is the cost. Not badness — unreliability. A knob that gives you an excellent answer four times out of five is unusable in a place where the fifth one reaches a customer unreviewed.

### Top-p

Temperature reshapes the whole distribution, tail included. **Top-p** takes a different approach: sort tokens by probability, keep adding them up until you reach `p`, and throw away everything else before sampling.

At `top_p=0.8` the model samples only from the smallest group of tokens that covers 80% of the probability mass. The long tail of bad options is not made unlikely — it is removed from consideration entirely.

The two are usually used together: temperature controls how adventurous the sampler is, top-p puts a floor under how bad its worst pick can be.

In [ ]:
# Cell C8 — Same high temperature, with and without a nucleus cutoff
print("temperature 1.2, top_p 1.00  (full vocabulary in play)")
print(" ", generate(task, max_new_tokens=50, do_sample=True, temperature=1.2, top_p=1.0))
print()
print("temperature 1.2, top_p 0.80  (tail cut off)")
print(" ", generate(task, max_new_tokens=50, do_sample=True, temperature=1.2, top_p=0.8))

#### 🤔 Your turn

You are picking defaults for two features shipping next month:

1. A support assistant that answers billing questions from a knowledge base.
2. A tool that suggests taglines for a marketing campaign.

What temperature do you set for each, and what is the failure mode you are protecting against?

<details>
<summary>How I would argue it</summary>

**Support: low, 0 to 0.3, and honestly just use greedy.** The failure that ends careers here is a confident wrong answer about someone's money. You want reproducibility so that when a customer complains, you can replay the exact interaction. Variety has no value; two customers with the same question should get the same answer.

**Taglines: high, 0.9 to 1.2, and generate ten of them.** Here the failure mode is *sameness*. A human is going to filter the output anyway, so an occasional dud costs nothing while a boring-but-safe suggestion costs you the whole feature.

The general rule: ask who catches the mistake. If a human reviews every output, buy variety. If the output goes straight to a user or into another system, buy determinism.

</details>

### One more contract: chat templates

An instruct model is not a text-completion model. It was fine-tuned on conversations wrapped in a specific format, with special tokens marking where each speaker starts and stops. For Qwen those markers are `<|im_start|>` and `<|im_end|>`.

Send it a bare string and you have broken the contract. It does not know who is talking or that it is supposed to answer you. Sometimes you get a usable response anyway, which is worse than failing outright, because it means the bug survives your testing and shows up in production as "the model ignores the system prompt sometimes."

`tokenizer.apply_chat_template()` takes your list of `{"role", "content"}` dicts and produces the exact string the model was trained on. Every model family has its own format, and the template ships with the tokenizer, which is why you should never hand-roll this.

In [ ]:
# Cell C9 — The same question, three ways
raw_prompt = "What is quantization?"

user_only = tokenizer.apply_chat_template(
    [{"role": "user", "content": raw_prompt}],
    tokenize=False, add_generation_prompt=True,
)

with_system = tokenizer.apply_chat_template(
    [{"role": "system", "content": "You are an LLM deployment expert."},
     {"role": "user", "content": raw_prompt}],
    tokenize=False, add_generation_prompt=True,
)

print("1. Raw string, what a naive caller sends:")
print("  ", repr(raw_prompt))
print("\n2. Template with NO system message supplied:")
print("  ", repr(user_only))
print("\n3. System + user, what production sends:")
print("  ", repr(with_system))

Look at the structure in versions 2 and 3: `<|im_start|>role`, content, `<|im_end|>`, repeated, and then a trailing `<|im_start|>assistant` with nothing after it. That trailing fragment is `add_generation_prompt=True` at work. It is the model's cue that it is now its turn to speak.

Those markers are real entries in the vocabulary, not decoration.

**Now look at output 2 again, closely.** You passed one user message and nothing else. The template gave you back a system message you never wrote:

```
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
```

Qwen's chat template inserts a default persona when you do not supply one. This is not documented anywhere you would think to look — it lives in the template string that ships with the tokenizer.

It is worth knowing about, because it is the sort of thing that produces a confusing bug report. Someone says the model "keeps claiming it is Qwen" or "ignores our brand voice," and the cause is a default that got silently added to a prompt nobody thought was empty. **Always send an explicit system message**, even a short one, so you know exactly what the model was told.

Now watch what happens when you skip the template entirely.

In [ ]:
# Cell C10 — Raw string versus templated, actually generated
raw_inputs = tokenizer(raw_prompt, return_tensors="pt")
with torch.no_grad():
    raw_out = model.generate(**raw_inputs, max_new_tokens=50, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
print("NO TEMPLATE:")
print(" ", tokenizer.decode(raw_out[0][raw_inputs["input_ids"].shape[1]:], skip_special_tokens=True))

print("\n" + "-" * 60 + "\n")

fmt_inputs = tokenizer(user_only, return_tensors="pt")
with torch.no_grad():
    fmt_out = model.generate(**fmt_inputs, max_new_tokens=50, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
print("WITH TEMPLATE:")
print(" ", tokenizer.decode(fmt_out[0][fmt_inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Read both carefully, because the difference is subtler than you might expect and that is exactly the problem.

The untemplated version is **continuing a document**, not answering a person. Watch where it goes after the first sentence: it starts expanding on a tangent ("Sampling involves dividing the continuous...") the way an encyclopedia entry would, because as far as it knows it was handed a fragment of text to extend. The templated version knows it was asked a question and stops when it has answered.

Here is the trap. Both outputs look plausible. If you built a feature on the untemplated path, it would pass a casual review, and then it would fail in the ways that are hardest to diagnose: your system prompt ignored, responses that trail into unrelated territory, formatting instructions that work sometimes. A loud error would be kinder. Use the template.

#### ✅ Checkpoint C

- [ ] Greedy gave you `Identical? True`; sampling gave you `False`. You can say why.
- [ ] Temperature divides the logits before the softmax. You saw the distribution change shape.
- [ ] Top-p removes the tail; temperature reshapes everything.
- [ ] You can point at `<|im_start|>` in the output and say what it is for.
- [ ] You know which knob you would ship for a support bot, and why.

---

# Part D — Multi-turn chat

**~13 minutes**

## The uncomfortable fact

A chat application does not have memory. The model has no state between calls. It does not know you asked it something thirty seconds ago.

What looks like memory is something you build: on every single turn, you resend the **entire conversation** and hope it fits.

```
Turn 1:  [system] [user: "What is quantization?"]
         → model → "Quantization reduces..."

Turn 2:  [system] [user: "What is quantization?"] [assistant: "Quantization reduces..."]
         [user: "How does it compare to pruning?"]
         → model → "Pruning removes..."

Turn 3:  [system] [everything above] [user: "Which should I try first?"]
         → model → "..."
```

Two things follow, and both cost money:

1. **You pay for the whole history, every turn.** Turn 10 re-sends turns 1 through 9. On a hosted API that is input tokens you are billed for repeatedly.
2. **The history eventually does not fit.** And when it does not, things break quietly.

We are going to build this by hand first. Four cells, one turn. Then we will do it a second time, and by the end of the second time you will have an opinion about how the code should be organized.

## Turn one, step by step

In [ ]:
# Cell D1 — The history is a list. That is the whole data structure.
history = [
    {"role": "system", "content": "You are an LLM deployment expert. Answer in at most two sentences."}
]

print(history)

In [ ]:
# Cell D2 — Step 1 of 4: add what the user said
history.append({"role": "user", "content": "What is quantization?"})

for message in history:
    print(f"{message['role']:>9} : {message['content']}")

In [ ]:
# Cell D3 — Steps 2 and 3 of 4: format the whole history, then generate
formatted = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt")
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

reply = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

print(reply)
print(f"\n[sent {prompt_len} tokens to get that]")

In [ ]:
# Cell D4 — Step 4 of 4: write the reply back into the history
history.append({"role": "assistant", "content": reply})

print(f"Messages in history: {len(history)}")

That last cell is one line and it is the one people forget.

Leave it out and the model never finds out what it said. Turn two would show it your first question and your second question back to back, with no answer in between, and it would either answer the first one again or produce something incoherent. Every "my chatbot has amnesia" bug is some version of a missing append.

## Turn two

Same four steps. All of it, again.

In [ ]:
# Cell D5 — Turn two: the identical four steps
history.append({"role": "user", "content": "How does it compare to pruning?"})          # step 1

formatted = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)   # step 2
inputs = tokenizer(formatted, return_tensors="pt")
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():                                                                    # step 3
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
reply = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

history.append({"role": "assistant", "content": reply})                                  # step 4

print(reply)
print(f"\n[sent {prompt_len} tokens this time, and the history is now {len(history)} messages]")

Compare the token count in D5 against D3. It went up, and you did not write a longer question. The growth is the history being re-sent.

## So why a class?

Look at what you just did twice.

You copied four steps. Two of them (`append` the user message, `append` the assistant reply) exist only to keep `history` correct, and if you forget either one the bug is silent. The other two do the actual work and need `history` to be in the right state before they run.

That is the tell. **You have data that must stay consistent, and operations that are the only things allowed to touch it.** A plain function would mean passing `history` in and getting it back out on every call, with the caller still responsible for the appends — which means the caller can still forget.

Put them together and forgetting becomes impossible:

- `self.history` is the data.
- `chat()` is the only thing that touches it, and it does all four steps or none.

That is not object-oriented dogma. It is the observation that these four steps have to happen together, and nothing else should be able to do three of them.

The class below is the code you already wrote, with the repetition removed. Read it and find the four steps.

In [ ]:
# Cell D6 — The same four steps, wrapped so you cannot get them out of order
class ChatSession:
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.history = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message, max_new_tokens=80):
        self.history.append({"role": "user", "content": user_message})            # step 1

        formatted = tokenizer.apply_chat_template(                                 # step 2
            self.history, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors="pt")
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():                                                      # step 3
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

        self.history.append({"role": "assistant", "content": reply})               # step 4
        return reply

    def token_count(self):
        formatted = tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        )
        return len(tokenizer(formatted)["input_ids"])

    def show_prompt(self):
        print(tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        ))


print("ChatSession defined. No output yet -- the next cell uses it.")

Every chat framework you will ever use is this object with more features bolted on: retries, streaming, tool calls, token budgets, persistence to a database. LangChain calls it memory. The OpenAI Assistants API calls it a thread. Underneath, all of them are appending dicts to a list and re-sending it.

Now three turns in five lines.

In [ ]:
# Cell D7 — Three turns, watching the history grow
bot = ChatSession("You are an LLM deployment expert. Answer in at most two sentences.")

for question in ["What is quantization?",
                 "How does it compare to pruning?",
                 "Which should I try first on a 7B model?"]:
    print(f">>> {question}")
    print(bot.chat(question))
    print(f"    [history is now {bot.token_count()} tokens]\n")

Watch that token count climb. Each turn re-sends everything before it, so the cost of turn N includes turns 1 through N-1 all over again.

You may also notice the answers are shaky. Mine defined quantization as "dividing into smaller intervals (quantiles)," which is wrong — quantization reduces numerical *precision*, it has nothing to do with quantiles. It also blew straight through the two-sentence limit in the system prompt.

Both of those are worth seeing. A 0.5B model has weak factual grounding and weak instruction-following, and the bigger models you deploy have the same failure modes in subtler forms. The mechanism you are studying here is identical at 70B. The quality is not.

Here is the part that makes it real. This is the literal string that went to the model on the last turn.

In [ ]:
# Cell D8 — The actual prompt behind that last one-sentence answer
bot.show_prompt()

Read it properly. All three questions are in there. All three answers are in there. The system prompt is in there. Every `<|im_start|>` and `<|im_end|>` is in there.

The model was handed all of that just to answer one short follow-up question. It did not remember the conversation — it re-read the conversation, from the top, the way it would read any other document.

That is what "stateless" means in practice, and it is why context management is a real engineering concern rather than a detail.

In [ ]:
# Cell D9 — How close are we to the limit?
limit = config.max_position_embeddings
used = bot.token_count()

print(f"Used      : {used:,} tokens")
print(f"Limit     : {limit:,} tokens")
print(f"Remaining : {limit - used:,} tokens")
print(f"Pressure  : {used / limit:.2%}")

Well under one percent. Three turns of a toy conversation is nothing.

Now do the arithmetic for something real. A support bot averaging 200 tokens a turn, running 50 turns, is 10,000 tokens. Add a detailed system prompt and a few retrieved documents per turn from your RAG pipeline, and 32,768 stops looking generous.

**What happens when you hit the wall is the part worth remembering: usually nothing visible.** No exception, no warning. Depending on the stack, the oldest tokens get dropped or the request is truncated, and the model simply stops being able to see the beginning of the conversation. Your users describe it as "it forgot what I told it," and your logs show a series of perfectly successful API calls.

### The four ways out

| Strategy | What it does | What it costs you |
|---|---|---|
| **Sliding window** | Keep the system prompt and the last N turns, drop the rest | Anything older is gone. Simple, and brutal. |
| **Summarize** | Compress old turns into a running summary | An extra model call per compaction, and lossy in ways you cannot predict |
| **Retrieve** | Put old turns in a vector store, pull back only what is relevant | Infrastructure, latency, and a retrieval quality problem (Lab 6) |
| **Fresh session** | Start over, carry a handoff summary forward | Obvious to the user, and sometimes that honesty is the right call |

There is no free option. Every one of them trades away continuity, money, or latency. Pick deliberately.

#### ✅ Checkpoint D

- [ ] Why does the token count go up when your questions stay the same length?
- [ ] Which single line, if deleted, gives your bot amnesia?
- [ ] What does the model actually receive on turn three?
- [ ] What does a full context window look like in your logs?

<details>
<summary>Answers</summary>

**Because the whole history is re-sent every turn.** Your question is a small addition to a prompt that already contains everything that came before.

**`self.history.append({"role": "assistant", ...})`** — step 4. Without it the model never sees its own replies.

**Every message, formatted with the special tokens,** concatenated into one string, as cell D8 showed.

**Like success.** HTTP 200s, no errors, users reporting that the assistant forgot something. Silent truncation is the failure mode.

</details>

---

## ✏️ Exercise — Write the sliding window

Your turn to write code.

`trim` should keep the system message plus the last `max_turns` user/assistant **pairs**, and drop everything older. A "turn" is two messages: one user, one assistant.

Fill in the body below, then run the cell under it to check your work.

In [ ]:
# Cell E1 — Your implementation
def trim(history, max_turns=1):
    system_message = history[:1]        # always keep the system prompt
    conversation = history[1:]          # everything else is user/assistant pairs

    # TODO: return system_message plus only the last `max_turns` pairs of `conversation`.
    # Hint: each turn is 2 messages, and negative list slicing takes from the end.

    return history                      # <- replace this

In [ ]:
# Cell E2 — Check your work
print(f"Before : {len(bot.history)} messages, {bot.token_count()} tokens")

bot.history = trim(bot.history, max_turns=1)

print(f"After  : {len(bot.history)} messages, {bot.token_count()} tokens")
print()
for message in bot.history:
    preview = message["content"][:70].replace("\n", " ")
    print(f"{message['role']:>9} : {preview}...")

**You got it right if:** the history drops to 3 messages (system, the last question, the last answer), the token count falls by well over half, and the two earlier turns are gone from the printout.

<details>
<summary>Solution</summary>

```python
def trim(history, max_turns=1):
    system_message = history[:1]
    conversation = history[1:]
    return system_message + conversation[-max_turns * 2:]
```

Two things to notice about this three-line function:

It assumes the conversation is a clean sequence of alternating user/assistant pairs. Add a tool call, a retry, or a system message injected mid-conversation and the arithmetic quietly breaks — you can end up starting your window on an assistant message, which some models handle badly.

It also keeps a fixed number of *turns*, not a fixed number of *tokens*. One user pasting a long log file blows your budget even though the turn count looks fine. Production implementations trim on token count for exactly this reason.

</details>

#### 🤔 One last question

You just trimmed the history to a single turn. Ask the bot "what were we talking about earlier?" — what does it say, and why?

<details>
<summary>Answer</summary>

It has no idea. Those messages are not "forgotten," they were **deleted from a Python list**. The model has no other copy of them and no memory outside the prompt you build.

This is the same stateless fact from Lab 2, arriving with consequences attached. The sliding window is not a way to help the model remember more. It is you choosing what the model is allowed to know, on every single turn.

</details>

---

## ✅ Lab 3 complete

You ran all of this yourself:

- [ ] Read a real config and said what each number means for deployment
- [ ] Found GQA in the weight shapes (`q_proj` 896, `k_proj` 128) before anyone told you it was there
- [ ] Saw the KV cache tensor and identified the `2` as the key/value head count
- [ ] Measured generation with and without the cache
- [ ] Turned that into a concurrent-user estimate
- [ ] Looked at a next-token probability distribution, then watched temperature reshape it
- [ ] Proved greedy is deterministic and sampling is not
- [ ] Built a conversation turn by hand, then refactored it into a class for a stated reason
- [ ] Printed the full prompt behind a one-sentence answer
- [ ] Wrote a sliding-window trim and named what it costs

## What to take with you

1. **The config is a datasheet.** Layers and hidden size tell you what the model costs to hold. `num_key_value_heads` tells you what it costs to *serve*, which is the number that usually decides things.
2. **GQA is why long context is affordable.** 14 query heads sharing 2 key/value heads cuts the KV cache sevenfold at nearly no quality cost. Every serious open model does this now.
3. **The KV cache is load-bearing.** You measured the alternative. Memory for compute, and the memory scales with users × context.
4. **Sampling knobs are product decisions.** Determinism is a feature when something downstream has to parse the output. Variety is a feature when a human filters it.
5. **Chat templates are a contract.** Break it and you get plausible-looking output that ignores your system prompt.
6. **Memory is an illusion you pay for.** Every turn re-sends everything. Context management is a design decision you make up front, not a problem you discover in production.

## Stretch goals

1. **Compare before you commit.** Load configs only (no weights, it is instant) for `mistralai/Mistral-7B-Instruct-v0.3` and `google/gemma-2-2b-it`. Compute KV cache bytes per token for each using the formula from cell A7. Which would you self-host for a 4,000-token RAG prompt serving 50 concurrent users?
2. **Trim on tokens, not turns.** Rewrite `trim` to drop the oldest turns until `token_count()` falls under a budget you pass in. This is what production actually does.
3. **Watch the distribution collapse.** Take cell C3 and add `T=0.01` and `T=3.0`. At what temperature does the top token stop dominating? At what point does the distribution become effectively uniform?
4. **Force the wall.** Give a `ChatSession` a deliberately tiny budget by trimming to a 200-token limit, run eight turns, and watch what the bot can and cannot recall. Then decide which of the four strategies you would reach for.

## Next

[Lab 4 — Quantize + LoRA](../04_Quantize_LoRA/README.md) needs a **T4 GPU**, so enable it before you open the notebook.

You estimated INT4 memory in cell A3 and concurrent users in cell B4. Lab 4 puts a bigger Qwen (1.5B) on real hardware, measures FP16 against NF4 for actual, and then trains a small LoRA adapter on top. The arithmetic you did here becomes a number on a GPU.